# HookLink2iOS — Ocarina of Time (Switch 2 Edition)

Notebook adaptado do Link2iOS_BOTW: reutiliza a autenticação OAuth do Colab e a pasta Google Drive já configurada no projeto. Não solicita novamente ID da pasta.

**Alvo:** Title ID `0400486024DEC000`, update `v131072`, GPU preset inicial `g18p`.

A entrada deve ser um NUP ou um arquivo compactado que contenha exatamente um NUP. A pasta raiz de trabalho no Drive é a mesma do Link2iOS. O notebook preserva a autenticação da conta Google já usada no Colab.


In [ ]:
%cd /content
!apt-get update -qq
!apt-get install -y -qq libarchive-tools
!python -m pip install --quiet requests google-api-python-client google-auth backports.zstd
!rm -rf /content/HookLink2iOS-repo
!git clone --depth 1 --branch HookLink2iOS https://github.com/joao9vegeto-stack/KintTany.git /content/HookLink2iOS-repo

from pathlib import Path
import sys
sys.path.insert(0, "/content/HookLink2iOS-repo/colab")
from hooklink2ios_prepare import prepare

print("HookLink2iOS checkout pronto.")


## 1. Autenticação Google Drive (reaproveitada do Link2iOS)

Usa a pasta já configurada no notebook original. Não é necessário informar outro ID nem criar conta de serviço.


In [ ]:
from google.colab import auth
import google.auth
from googleapiclient.discovery import build

DRIVE_FOLDER_ID = "19HXEJCK-26NfBVKeUP3wqFxMFpoF3ykm"
auth.authenticate_user()
DRIVE_CREDENTIALS, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/drive"])
drive = build("drive", "v3", credentials=DRIVE_CREDENTIALS, cache_discovery=False)
drive_parent = drive.files().get(fileId=DRIVE_FOLDER_ID, fields="id,name,mimeType").execute()
assert drive_parent.get("mimeType") == "application/vnd.google-apps.folder", drive_parent
print("Google Drive autenticado:", drive_parent.get("name"))
print("Pasta reutilizada:", DRIVE_FOLDER_ID)


## 2. Selecionar a entrada do jogo

Pode ser um arquivo NUP/compactado já armazenado na pasta do projeto ou um URL HTTPS próprio. O notebook reutiliza a pasta raiz já existente no Drive; não solicita outro destino.


In [ ]:
from getpass import getpass

SOURCE_MODE = input("Origem [drive/url]: ").strip().lower()
SOURCE_DRIVE_FILE_ID = ""
SOURCE_URL = ""
SOURCE_FILENAME = ""
if SOURCE_MODE == "drive":
    SOURCE_DRIVE_FILE_ID = input("ID do arquivo NUP/RAR/ZIP/7z no Drive: ").strip()
elif SOURCE_MODE == "url":
    SOURCE_URL = getpass("URL HTTPS do arquivo: ").strip()
    SOURCE_FILENAME = input("Nome do arquivo com extensão: ").strip()
else:
    raise ValueError("Escolha 'drive' ou 'url'.")

GITHUB_PAT = getpass("GitHub Fine-grained PAT (Contents: Read/Write; Actions: Read) — entrada oculta: ").strip()
assert GITHUB_PAT, "PAT obrigatório para handoff e recuperação do artifact."


## 3. Preparar RomFS no Linux e iniciar somente HookLink2iOS

O Colab cria uma pasta de execução dentro da pasta já existente, envia NUP, RomFS e overlay, valida os hashes e atualiza somente `hooklink2ios/handoff.json` na branch `HookLink2iOS`. O workflow gera a IPA como artifact. A transferência final para o Drive reutiliza a mesma autenticação OAuth do Colab, sem exigir uma nova conta de serviço no Actions.


In [ ]:
RESULT = prepare(
    source_url=SOURCE_URL,
    source_filename=SOURCE_FILENAME,
    source_drive_id=SOURCE_DRIVE_FILE_ID,
    drive_parent_folder_id=DRIVE_FOLDER_ID,
    drive=drive,
    github_token=GITHUB_PAT,
    repo_root=Path("/content/HookLink2iOS-repo"),
)

HANDOFF = RESULT["handoff"]
print("Preparação Linux concluída.")
print("Pasta de execução:", HANDOFF["drive"]["run_folder_url"] or HANDOFF["drive"]["run_folder_id"])
print("NUP:", HANDOFF["nup"]["size"], "bytes")
print("RomFS:", HANDOFF["romfs"]["size"], "bytes")
print("Overlay:", HANDOFF["resolution_overlay"]["size"], "bytes")
print("GitHub Actions:", RESULT["github"]["actions_url"])
